Для інсталювання необхідного пакету в середовищі Anaconda потрібно ввести в терміналі Anaconda:

```python
pip install pulp
pip install numpy
```

Для інсталювання необхідного пакету в середовищі Google Colab потрібно ввести першій комірці:

```python
!pip install pulp
!pip install numpy
```

Більше деталей за [посиланням](https://colab.research.google.com/notebooks/snippets/importing_libraries.ipynb).

## Дано

Нехай маємо математичну модель траспортної задачі лінійного програмування (ТЗЛП):

$$\begin{array}{rcl} f = \min\limits_{x \in X} \{ 7x_{11} + 3x_{12} + 5x_{13} + 4x_{14} + 2x_{15} + \\
+ 6x_{21} + 2x_{22} + 3x_{23} + x_{24} + 7x_{25} + \\
+ 3x_{31} + 5x_{32} + 2x_{33} + 6x_{34} + 4x_{35} \}, \\
\end{array}$$

$$\begin{cases}
\begin{aligned}
x_{11} + x_{12} + x_{13} + x_{14} + x_{15} = 40; \\
x_{21} + x_{22} + x_{23} + x_{24} + x_{25} = 150; \\
x_{31} + x_{32} + x_{33} + x_{34} + x_{35} = 40, \\
\end{aligned}
\end{cases}$$

$$\begin{cases}
\begin{aligned}
x_{11} + x_{21} + x_{31} = 20; \\
x_{12} + x_{22} + x_{32} = 80; \\
x_{13} + x_{23} + x_{33} = 90; \\
x_{14} + x_{24} + x_{34} = 60; \\
x_{15} + x_{25} + x_{35} = 40, \\
\end{aligned}
\end{cases}$$

$$\begin{array}
1x_{ij} \geq 0, \quad i = \overline{1,3}, \quad j = \overline{1,5}.
\end{array}$$

## Блок 1. Оголошення та ініціалізація вхідних даних

In [17]:
from pulp import *
import numpy as np

In [18]:
# Склади та торгівельні точки: List
Warehouse = ['A1', 'A2', 'A3']
Branch = ['B1', 'B2', 'B3', 'B4', 'B5']

In [19]:
# Об'єми вивозу
demand = {'A1' : 40, 'A2' : 150, 'A3' : 100}

In [20]:
# Об'єми поставок
supply = {'B1' : 20, 'B2' : 80, 'B3' : 90, 'B4' : 60, 'B5' : 40}

In [21]:
# Витрати на перевезення одиниці вантажу із складів до торгівельних точок
cost = {'A1': {'B1' : 7, 'B2' : 3, 'B3' : 5, 'B4' : 4, 'B5' : 2},
        'A2': {'B1' : 6, 'B2' : 2, 'B3' : 3, 'B4' : 1, 'B5' : 7},
        'A3': {'B1' : 3, 'B2' : 5, 'B3' : 2, 'B4' : 6, 'B5' : 4},
       }

## Блок 2. Реалізація транспортної задачі лінійного програмування

In [22]:
# Створення об'єкту математичної моделі
probTrans = LpProblem("ТЗЛП", LpMinimize)

In [23]:
# Оголошення матриці, що моделює всі можливі шляхи перевезення 
routes = [(i,j) for i in Warehouse for j in Branch]
print(routes)

[('A1', 'B1'), ('A1', 'B2'), ('A1', 'B3'), ('A1', 'B4'), ('A1', 'B5'), ('A2', 'B1'), ('A2', 'B2'), ('A2', 'B3'), ('A2', 'B4'), ('A2', 'B5'), ('A3', 'B1'), ('A3', 'B2'), ('A3', 'B3'), ('A3', 'B4'), ('A3', 'B5')]


In [24]:
# Оголошення та ініціалізація змінних математичної моделі
amount_vars = LpVariable.dicts(name="X", indexs=(Warehouse,Branch), lowBound=0, upBound=None)
print(amount_vars)

{'A1': {'B1': X_A1_B1, 'B2': X_A1_B2, 'B3': X_A1_B3, 'B4': X_A1_B4, 'B5': X_A1_B5}, 'A2': {'B1': X_A2_B1, 'B2': X_A2_B2, 'B3': X_A2_B3, 'B4': X_A2_B4, 'B5': X_A2_B5}, 'A3': {'B1': X_A3_B1, 'B2': X_A3_B2, 'B3': X_A3_B3, 'B4': X_A3_B4, 'B5': X_A3_B5}}


C:\Users\radiu\AppData\Local\Temp/ipykernel_19292/774953585.py:2: DeprecationWarning: 'indexs' is deprecated; use 'indices'.
  amount_vars = LpVariable.dicts(name="X", indexs=(Warehouse,Branch), lowBound=0, upBound=None)


In [25]:
# Оголошення та ініціалізація цільової функції
# lpSum - обчислення суми добутків C*X
probTrans += lpSum(amount_vars[i][j]*cost[i][j] for (i,j) in routes)

In [26]:
# Оголошення та ініціалізація обмежень моделі
for i in Warehouse:
    probTrans += lpSum(amount_vars[i][j] for j in Branch) == demand[i]
    
for j in Branch:
    probTrans += lpSum(amount_vars[i][j] for i in Warehouse) == supply[j]

In [27]:
# Відображення математичної моделі
print(probTrans)

ТЗЛП:
MINIMIZE
7*X_A1_B1 + 3*X_A1_B2 + 5*X_A1_B3 + 4*X_A1_B4 + 2*X_A1_B5 + 6*X_A2_B1 + 2*X_A2_B2 + 3*X_A2_B3 + 1*X_A2_B4 + 7*X_A2_B5 + 3*X_A3_B1 + 5*X_A3_B2 + 2*X_A3_B3 + 6*X_A3_B4 + 4*X_A3_B5 + 0
SUBJECT TO
_C1: X_A1_B1 + X_A1_B2 + X_A1_B3 + X_A1_B4 + X_A1_B5 = 40

_C2: X_A2_B1 + X_A2_B2 + X_A2_B3 + X_A2_B4 + X_A2_B5 = 150

_C3: X_A3_B1 + X_A3_B2 + X_A3_B3 + X_A3_B4 + X_A3_B5 = 100

_C4: X_A1_B1 + X_A2_B1 + X_A3_B1 = 20

_C5: X_A1_B2 + X_A2_B2 + X_A3_B2 = 80

_C6: X_A1_B3 + X_A2_B3 + X_A3_B3 = 90

_C7: X_A1_B4 + X_A2_B4 + X_A3_B4 = 60

_C8: X_A1_B5 + X_A2_B5 + X_A3_B5 = 40

VARIABLES
X_A1_B1 Continuous
X_A1_B2 Continuous
X_A1_B3 Continuous
X_A1_B4 Continuous
X_A1_B5 Continuous
X_A2_B1 Continuous
X_A2_B2 Continuous
X_A2_B3 Continuous
X_A2_B4 Continuous
X_A2_B5 Continuous
X_A3_B1 Continuous
X_A3_B2 Continuous
X_A3_B3 Continuous
X_A3_B4 Continuous
X_A3_B5 Continuous



In [28]:
# Розв'язок ТЗЛП
probTrans.solve(PULP_CBC_CMD(msg=1))

1

## Блок 3. Виведення результату розв'язку транспортної задачі лінійного програмування

In [29]:
# Виведення результату оптимізації
print("Status:", LpStatus[probTrans.status])

Status: Optimal


In [33]:
# Виведення значень перевезень
for v in probTrans.variables():
    if v.varValue > 0:
        print(v.name, "=", v.varValue)

X_A1_B5 = 40.0
X_A2_B2 = 80.0
X_A2_B3 = 10.0
X_A2_B4 = 60.0
X_A3_B1 = 20.0
X_A3_B3 = 80.0


In [31]:
print("Оптимальне значення цільової функції: f_min =", value(probTrans.objective))

Оптимальне значення цільової функції: f_min = 550.0


In [32]:
outputPlan = []

for v in probTrans.variables():
    outputPlan.append(int(v.varValue))

outputPlan = np.array(outputPlan)
outputPlan = outputPlan.reshape(len(Warehouse),len(Branch))
print("Матричне виведення результату:")
print(outputPlan)

Матричне виведення результату:
[[ 0  0  0  0 40]
 [ 0 80 10 60  0]
 [20  0 80  0  0]]
